# LRCN Training Replica

This notebook documents the current LRCN training workflow used for the KTH action recognition project. It follows the same structure as the main training file and explains each step in a more readable form.

The goal is to prepare video clips from the KTH dataset, build a dataset for training, define an LRCN model, and then prepare the model for optimization.

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import Counter
import re

## 1. Dataset path and class setup

The project uses the KTH dataset directories. The data path is resolved so that it works whether the files are stored directly under the project root or inside a Dataset folder.

In [ ]:
base_dir = Path(r"C:\NNDL Lab Proj\KTH")
dataset_path = base_dir / "Dataset" if (base_dir / "Dataset").exists() else base_dir

classes = [
    "walking",
    "running",
    "handwaving",
    "handclapping"
]

print("Dataset path:", dataset_path)
print(classes)

## 2. Collect all video paths and labels

Each video is assigned a numeric label according to its action class. This creates the raw dataset used for training and evaluation.

In [ ]:
video_paths = []
labels = []

for label, class_name in enumerate(classes):
    class_path = dataset_path / class_name

    for video_file in class_path.glob("*.avi"):
        video_paths.append(video_file)
        labels.append(label)

print("Total videos:", len(video_paths))
print("Total labels:", len(labels))

## 3. Person splitting for train/validation/test

The dataset is split by person ID, which is a common way to avoid leakage between train and test data in action recognition tasks.

In [ ]:
person_ids = []

for path in video_paths:
    match = re.search(r"person(\d+)", path.name)
    person_ids.append(int(match.group(1)))

unique_persons = sorted(set(person_ids))
print("Unique persons:", unique_persons)

train_persons = [11, 12, 13, 14, 15, 16, 17, 18]
val_persons = [19, 20, 21, 23, 24, 25, 1, 4]
test_persons = [22, 2, 3, 5, 6, 7, 8, 9, 10]

train_videos = []
train_labels = []
val_videos = []
val_labels = []
test_videos = []
test_labels = []

for video_path, label, person_id in zip(video_paths, labels, person_ids):
    if person_id in train_persons:
        train_videos.append(video_path)
        train_labels.append(label)
    elif person_id in val_persons:
        val_videos.append(video_path)
        val_labels.append(label)
    elif person_id in test_persons:
        test_videos.append(video_path)
        test_labels.append(label)

print("Training videos:", len(train_videos))
print("Validation videos:", len(val_videos))
print("Testing videos:", len(test_videos))

In [ ]:
print("TRAIN:", Counter(train_labels))
print("VALIDATION:", Counter(val_labels))
print("TEST:", Counter(test_labels))

## 4. Build a custom dataset

The dataset class reads each video, samples 16 frames with a fixed step, converts them to RGB, resizes them, normalizes them, and returns a sequence tensor of shape (T, C, H, W).

In [ ]:
class KTHDataset(Dataset):

    def __init__(self, video_paths, labels, num_frames=16, frame_step=6, image_size=112):
        self.video_paths = video_paths
        self.labels = labels
        self.num_frames = num_frames
        self.frame_step = frame_step
        self.image_size = image_size

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, index):
        video_path = self.video_paths[index]
        label = self.labels[index]

        cap = cv2.VideoCapture(str(video_path))
        frames = []

        sample_indices = np.arange(0, self.num_frames * self.frame_step, self.frame_step)

        for frame_index in sample_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ret, frame = cap.read()

            if not ret:
                cap.release()
                raise RuntimeError(f"Could not read frame {frame_index} from {video_path}")

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (self.image_size, self.image_size))
            frame = frame.astype(np.float32) / 255.0
            frame = torch.from_numpy(frame).permute(2, 0, 1)
            frames.append(frame)

        cap.release()
        video = torch.stack(frames)

        return video, torch.tensor(label, dtype=torch.long)

In [ ]:
train_dataset = KTHDataset(train_videos, train_labels)
video, label = train_dataset[0]

print("Video shape:", video.shape)
print("Label:", label)
print("Min/Max:", video.min().item(), video.max().item())

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
videos, labels = next(iter(train_loader))

print("Batch video shape:", videos.shape)
print("Batch label shape:", labels.shape)
print("Labels:", labels)

## 5. Define the CNN + LSTM model

The LRCN model first processes each frame with a CNN, then feeds the sequence of CNN features into an LSTM, and finally produces class logits.

In [ ]:
class CNNFeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return x


class LRCN(nn.Module):

    def __init__(self, num_classes=4):
        super().__init__()
        self.cnn = CNNFeatureExtractor()
        self.lstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        x = self.cnn(x)
        x = x.view(B, T, 128)
        output, (hidden, cell) = self.lstm(x)
        x = hidden[-1]
        x = self.fc(x)
        return x


model = LRCN(num_classes=4)
print(model)

In [ ]:
prediction = model(videos)
print("Input shape:", videos.shape)
print("Output shape:", prediction.shape)
print("Prediction:", prediction)

## 6. Training setup

The model is prepared for optimization with a cross-entropy loss and Adam optimizer.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

## Final interpretation

This notebook demonstrates the complete training preparation pipeline for the KTH LRCN project: dataset discovery, class mapping, train/test person split, temporal frame sampling, model definition, and preparation for optimization. It is the training-stage continuation of the project and connects the preprocessing workflow to the deep learning model.